# Proportional - Integrative - Derivative (PID) Controllers: Generalities

## Introduction

PID (proportional, integral, derivative) control is so widely applicable and successful, that it sometimes referred to as "the most successful technology of all times". A PID controller is an algorithm extensively used in industrial systems to generate a control signal over time $t$ based on a tracking error $e(t)$. 

The error is calculated as the difference between a desired setpoint value, usually referred to as the reference signal, $r(t)$, and a measured process variable, or, in other words, an output of the system $y_m(t)$. 

The goal of the controller is to create a command signal $u(t)$ that reduces the tracking error over time, ideally driving it to zero, so that the measured output coincides with the desired reference signal. The controller does so by driving the system actuators with an input derived determined by the simple sum of three terms: a proportional (to the tracking error), integral (to the tracking error), and derivative (to the tracking error) terms.

## Fundamental notions

Let's refresh some basic terminology and facts of PID control. 

### Terminology and Definitions

When referring to PID control, the following terms are typically used. 

* **Process Variable/Measured Output**: The parameter of the system that is being measured and controlled. Alternatively, this is referred to as the measured output, and indicated with $y_m(t)$, referring to standard control systems literature. For more information on instroduction to signals and system, check out this [Duckietown introduction to Control Systems](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/control/01-controls-intro.html).
* **Setpoint/Reference signal**: or reference signal, $r(t)$, is the desired value we are trying to drive the the process variable to.
* **Control Variable/Manipulated Variable**: The output of the controller that serves as input to the system in order to minimize error between the setpoint and the process variable. In control systems literature this is typically expressed as $u(t)$.
* **Steady-State Value**: The value of the process variable as time goes to infinity.
* **Steady-State Error**: The difference between the setpoint and the steady-state value.
* **Rise Time**: The time required for the process variable to rise from 10 percent to 90 percent of the steady-state value.
* **Settling Time**: The time required for the process variable to settle within a certain percentage of the steady-state value.
* **Overshoot**: The amount the process variable exceeds the setpoint (expressed as a percentage).

### General Algorithm

The error of the system $e(t)$, is ideally calculated as the difference between the setpoint $r(t)$ and the process variable, or system output, $y(t) = f(x,u,t)$ - where $f(\cdot)$ is a shortcut way to represent "whatever dynamic process is going on between the input and output of the system". 

Therefore, always ideally, the tracking error is:

$$e(t) = r(t) - y(t)$$

We are insisting on the "ideal" part because the _true_ value of the process variable is generally unknown, all we can do is measure it. Measuring implies the use of some instrument, which pretty much always introduces some kind noise. Truly, therefore, any practical PID implementation is driven by $e(t) = r(t) - y_m(t)$.

When tuned appropriately, the PID controller improves the _performance_ and _robustness_ of the closed loop response, by reducing the rise time and settling time of the system, eliminating steady-state error, instroducing some degree of disturbance rejection, and improving stability. It does so by changing the control variable $u(t)$ based on three control terms.

#### Proportional Term
The first control term is the proportional term, which produces an output that is proportional to $e(t)$:

$$P(t) = K_pe(t)$$

The magnitude of the proportional response is dependent upon $K_p \in \mathcal{R}$, which is the "proportional gain" constant. A higher proportional gain constant indicates a greater change in the controller's output in response to the system's error.

#### Integral Term

The integral term accounts for the accumulated tracking error over time. The output produced by this term is the sum of the instantaneous error over time, multiplied by the "integral gain constant" $K_i$:

$$I(t) = K_i\int_0^t\!e(\tau)\,\mathrm{d}\tau$$

#### Derivative Term

The derivative term is determined by the rate of change of the system's error over time, similarly multiplied by the "derivative gain constant" $K_d$:

$$D(t) = K_d\frac{de(t)}{dt}$$

#### Overall Control Function

The PID control function is the sum of the proportional, integral, and derivative terms:

$$u(t) = K_pe(t) + K_i\int_0^t\!e(\tau)\,\mathrm{d}\tau + K_d\frac{de(t)}{dt}$$

In practice, the *discretized* form of the control function is be more suitable for implementation:

$$u(t) = K_pe(t_k) + K_i\sum_{i=0}^k e(t_i)\Delta t + K_d\frac{e(t_k)-e(t_{k-1})}{\Delta t}$$

The figure below summarizes the inclusion of a PID controller within a basic control loop.

<figure>
    <img src="../assets/_images/pid/pid_controller_block_diagram.png" alt = "PID Control Block Diagram"  style="width:80%">
</figure>

### Tuning the PID controller parameters

Tuning a PID controller refers to iterating through the values of the $K_p$, $K_i$, and $K_d$ parameters to obtain a desireable control response. After understanding the general effects of each control term on the closed-loop (i.e., the system with the controller active) response, tuning can be accomplished through trial-and-error or by other specialized tuning schemes, such as the [Ziegler-Nichols tuning method](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method). 

Although the independent effects of each parameter are explained below, the three control terms may be correlated and so changing one parameter may impact the influence of another. The general effects of each term are therefore useful as reference, but the actual effects will vary depending on the specific dynamics of the underlying system, in other words, what is the exact form of what we previously referred to as $f(\cdot)$. 

Control literature - as we will soon be doing in the continuation of this notebook - tends to focus on explaning the effects ofo the parameters on the closed-loop system response assuming the controlled system is an LTI one, i.e., (i) linear, and (ii) time-invariant. Delving in the definition of LTI systems is beyond the scope of this LX (the curious reader can refer to a typical [Control System I introductory class on modeling and linearization](https://www.youtube.com/watch?v=GTiHWvx9N-4&list=PLP-dxWt_NmHtqCxuTAmzNVTPAgXossvUc&index=2&t=2174s)), but a few things are worth keeping in mind:

1. Real-world system are pretty much all **not** LTI systems, but rather nonlinear systems;
2. All nonlinear systems can be ("locally") approximated as LTI systems. 

In summary, none of what follows in fundamentally true, but it is close enough that it actually works (most of the times, with sufficient patience)! 

#### Effects of $K_p$

For a given level of error, increasing $K_p$ will proportionally increase the control output. This generally causes the system to react more quickly, decreasing the rise time and settling time by a small amount. On the other hand, increasing the proportional gain tends to cause overshoot, which in turn could destabilize the system. 

Increasing $K_p$ also has the effect of decreasing the steady-state error. However, as the value of the process variable approaches the setpoint and the error decreases, the proportional term will also decrease. As a result, with a P-controller (a controller with only the proportional term, i.e., ($K_i = K_d = 0$), the process variable will asymptotically approach the setpoint, but will never quite reach it. Thus, a P-controller cannot be used to completely eliminate steady-state error.

#### Effects of $K_i$

The integral term takes into account the history of the error, as well as its duration in time. The longer the error persists, the larger the integral term will grow, and eventually drive the error down. This term has the most notable effect of reducing and eliminating steady-state error. However, the build-up of error can cause the value of the process variable to overshoot, which can increase the settling time of the system, though it decreases the rise time. Moreover, implementing an integral term on a real system, through a computer or microcontroller, requires memory storage. 

#### Effects of $K_d$

By calculating the instantaneous rate of change of the system's error and using this slope for linear extrapolation, the derivative term somewhat anticipates future tracking error, making the sytem more reactive rather than proactive. 

While the proportional and integral terms both act to move the process variable to the setpoint, the derivative term rather dampens their efforts and decrease the amount the system overshoots. If appropriately tuned, the derivative term reduces oscillations, overshoot, settling time and improves the stability of the system. The derivative term has negligible effects on steady-state error and only decreases the rise time by a minor amount.

An **important caveat** of derivative terms is the following: *never take derivatives of noisy signals*. The most common additive white noise, which we have learned to be practically present in every real-world PID controller, will yield a practically infinite value when derived, leading to controller to unreasonable inputs, the system to instability, and potentially in things breaking and/or people getting hurt. Therefore, the best pratices in using derivative terms in real-world PID are:

1. always double-check the quality of the error signal before turning on the derivative term; 
2. Low-pass filter the signal if needed before sending it to the controller, or apply other averaging algorithms (e.g., moving windows are a popular alternative);
3. Alays start with very tiny $K_d$ values (tiny with respect to the other two terms in the controller), and verify that $D(t)$ is in the order of magnitude of $P(t)$ and $I(t)$ _before_ activating the controller on the real system/plant.

Finally, derivative (as well as integral) terms require, as their name suggests, derivatives (and integrals). This mathematical operation underlies the notion of infinitesimal, which is mostly a qualitative one. Computers tend to work better with quantitative inputs rather than qualitative guidelines, hence practically calculating these terms in a real-world scenario will lead to design choices.


### Calculating integrals in discrete time

The theory tells us that one of the component signals of the PID controller is proportional to the integral of the error over time:

$$ e_{int}(t) = k_p \int_0^t e(\tau) d\tau $$

An integral is an _infinite_ (pick the biggest number you can think of, _infinite_ is strictly more than that) **sum** of _infinitesimal_ (pick a positive number, the smallest you can think of, _infinitesimal_ is stricly smaller than that) bits, a concept that assumes _continuity_ of time. Continuity means that, for any two instant in time you can think about, arbitrarily close to each other, there will always be _infinite_ other instant between them.

But all computers in the world, including the one running on the Duckiebot, don't know how to do infinite or infinitesimal. "Think of the smallest" and "think of the biggest" are qualitative concepts that only humans can grasp.

Computers can do finite (instead of infinite or infinitesimal) though. Time, for a robot, is a sequence of instants. But when you make a computer take two consecutive instants, there is nothing in between. Computers have a notion of _discrete_ time, not continuous time.

The immediate repercussion of this fundamental limitation of computers is that we cannot really calculate integrals.

What we can do is have them calculate a _finite sum_ to approximate the actual integral. 

$$ e_{int}(t) = k_p \int_0^t e(\tau) d\tau \simeq \sum_{i=0}^{k} e_i \Delta t = (e_0 + e_1 + \dots + e_{k-1} + e_k)\Delta t$$

Where in the above approximation we assumed for simplicity that all time instants are equally spaced by a constant _time step_ $\Delta t$.

So how do we implment this integral component on our Duckiebots? We can note that:

$$ e_{int,k}= (e_0 + e_1 + \dots + e_{k-1} + e_k)\Delta t = (e_0 + e_1 + \dots + e_{k-1})\Delta t + e_k\Delta t = e_{int,k-1} + e_k \Delta t.$$


### Calculating derivatives in discrete time

Derivatives are the tool math uses to measure change. As you might imagine, derivatives are tremendously important operations as so many things in the universe change in some way. One could argue that derivatives are the most important of operators, and in fact open the doors to a whole field of math called _calculus_.  

Time derivatives are defined as _the ratio of the difference of a function, evaluated at two infinitesimally close to each other instants, and the time difference between them_. 

In fancy words, derivatives are _limits of the incremental ratio_ functions. 

$$ \dot e_t = \frac{de_t}{dt} = \lim_{dt \rightarrow 0} \frac{e_{t+dt}-e_t}{dt}$$

Without getting in the details, that $\lim$ part means that the $dt$ time difference is a very, very, small (positive) number. How small? The smaller you can imagine it the better you understood the derivative operation. _At the limit, it's zero_.  

But computers cannot do this "at the limit", because it is a qualitative leap of the mind. This is a very human thing to do. Computer only know finite time. So they need to know how much is actually $dt$ equal to (0.001? 0.0000001? or maybe even smaller that that?).

Computer cannot do derivatives, they can do _finite differences_, which approximate derivatives. There are many different formulations of finite differences, the simplest is called "Euler backwards" method. It basically approximates the derivative as a difference of the current and the previous function evaluation, divided by the time step $\Delta T = t_{k}-t_{k-1}$. 

$$\frac{de_t}{dt} \simeq e_{der,k} = \frac{e_k - e_{k-1}}{\Delta T}.$$



### Summary

<figure>
    <img src="../assets/_images/pid/control_term_effects_table.png" alt = "Summary of PID controller terms effect"  style="width:80%">
</figure>


### Ziegler-Nichols Closed-Loop Tuning Method
Ziegler and Nichols developed [two techniques](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method) for tuning PID controllers: a closed-loop tuning method and an open-loop tuning method. With the closed-loop tuning method, the PID controller is initially turned into a P controller with $K_p$ set to zero. $K_p$ is slowly increased until the system exhibits stable oscillatory behavior, at which point it is denoted $K_u$, the ultimate or critical gain. As such, $K_u$ should be the smallest $K_p$ value that causes the control loop to have regular oscillations. The ultimate or critical period $T_u$ of the oscillations needs to be measured. Then, using the constants determined experimentally by Ziegler and Nichols, the controller gain values can be computed as follows:  

$$ K_p = 0.6K_u $$
$$ K_i = 2K_p / (T_u) $$
$$ K_d = K_p(T_u) / 8 $$

Although the Ziegler-Nichols method may yield initial tuning values that work relatively well, the system's control loop can be tuned further by adjusting the controller gain values based on the general effects of each control term as explained above.

### Potential Problems
In real-world applications, the PID controller exhibits issues that require modifications to the general algorithm. In certain situations, one may find that a P-Controller, PD-Controller (eliminating the integral term), or a PI-Controller (eliminating the derivative term) are more advantageous controllers for the system. Alternatively, different techniques can be employed to counteract the problems that may affect the usability of a control term.

#### Integral Windup
Integral wind-up occurs when, due to a large change in setpoint, the control output causes the system's actuator to become saturated. At this point, the integrated error between the process variable and the setpoint will continue to grow (because the actuator is at its limit and cannot drive the process variable any closer to the setpoint). In turn, the control output will continue to grow and will no longer have any effect on the system. When the setpoint finally changes and the error changes sign (meaning the new setpoint is now below the value of the process variable), the integral term will take a while to "unwind" all of the error that it has accumulated before producing a reverse control action that will move the process variable in the correct direction towards the setpoint.

There exist numerous ways to address integral wind-up. One way is to keep the integral term within predefined upper and lower bounds. Another way is to set the integral term to zero if the control output will cause the system's actuator to saturate. Yet another way is to reduce the integral term by a constant multiplied by the difference between the actual output and the commanded output. If the actuator is not saturated, then the difference between the actual and commanded output will be zero and will not affect the integral term. If the actuator is saturated, then the additional feedback in the control loop will drive the commanded output closer to the saturation limit. If the setpoint changes and causes the error to change sign, then the integral term will not need to unwind in order to produce an appropriate control action. Setpoint ramping &mdash; in which the setpoint is increased or decreased incrementally to reach the desired value &mdash; may also help prevent integral wind-up.

#### Derivative Noise
Since the derivative term is proportional to the change in error, it is consequently highly sensitive to noise (which would produce drastic changes in error). Using a low-pass filter on the derivative term, or finding the derivative of the process variable (as opposed to the error), or taking a weighted mean of previous derivative terms could help ensure that high-frequency noise does not cause the derivative term to adversely affect the control output.

### Cascaded Controllers
When multiple measurements can be used to control a single process variable, these measurements can be combined using a cascaded PID controller. In cascaded PID control, two PID controllers are used conjointly to yield a better control response. The output of the PID controller for the outer control loop determines the setpoint for the PID controller of the inner control loop. The outer loop controller controls the primary process variable of the system, while the inner loop controller controls a system parameter that tends to change more rapidly in order to minimize the error of the outer control loop. The two controllers have separate tuning values, which can be optimized for the part of the system that they control. This enables an overall better control response for the system as a whole.


# Activity


## The True Value and Error Curves
The figure below shows a true value curve for a PID controller. Draw the corresponding error curve for this graph. You can draw by hand and upload the picture. (Hint: refer to the error definition equation from before)

<figure>
  <img src="../assets/_images/pid/true_value_curve.png" alt="True value curve" style="width:60%">
  <figcaption>True Value Curve for A PID Controller. The orange dot line indicates the setpoint and the black line is the true value curve.</figcaption>
</figure>


## Explain an Effect

Answer the following questions (3-5 sentences each):
  * What will happen when the absolute value of $K_{p}$ is very large? What will happen when the absolute value of $K_{p}$ is very small?
  * Can $K_{p}$ be tuned such that the $P$ term stops oscillations? Why or why not?
  * Can the process variable stabilize at the setpoint (i.e. zero steady-state error) with only the $P$ term and the $D$ term? Why or why not?                                      

<!-- 
[ANSWERS]
    The rise time decreases when $K_{p}$ increases.
    The settling time increases when $K_{i}$ increases.
    The overshoot decreases when $K_{d}$ increases.                                   
-->   

Explain the following effects caused by $K_{p}$, $K_{i}$ and $K_{d}$ (3-5 sentences each). For example, here is a sample answer (though you do not need to follow the pattern):

  * [Q:] *The rise time decreases when $K_{d}$ increases.
  * [A:] *When $K_{d}$ increases, the error at time step $t+1$ decreases. This is because larger and larger $K_{d}$ results in larger and larger control signals at time step $t$. This drives the system to achieve a lower error at time step $t+1$. As the error at time step $t+1$ decreases, the slope of the true value curve increases. Since the slope increases, the rising time towards the setpoint should decrease (slightly).




## Start Tuning

When designing a PID controller, it is important to choose a good set of $K_{p}$, $K_{i}$, and $K_{d}$; poor choices can result in undesirable behavior. The graphs in the figure below illustrate behavior resulting from unknown sets of $K_{p}$, $K_{i}$, and $K_{d}$. In each graph, the orange dot line indicates the setpoint and the black line is the true value curve. For each graph, answer the following (1-2 sentences each):


1. Which term(s) went wrong, if any? In other words, which term(s) are too high or too low?
2. How can you correct the behavior?

<figure>
    <img src="../assets/_images/pid/tuning1.png" alt = "PID tuning option"  style="width:60%">
</figure>

<figure>
    <img src="../assets/_images/pid/tuning2.png" alt = "PID tuning option"  style="width:60%">
</figure>

<figure>
    <img src="../assets/_images/pid/tuning3.png" alt = "PID tuning option"  style="width:60%">
</figure>

<figure>
    <img src="../assets/_images/pid/tuning4.png" alt = "PID tuning option"  style="width:60%">
</figure>



